# What the version track shows

A history can name its version file. Maxitor's track then draws a large point where the graph changed and a small one where only the version did. This notebook runs a real port and Maxitor over a small repository and reads what the track reads.

## Setup

In [ ]:
from __future__ import annotations

import contextlib
import io
import json
import socket
import subprocess
import tempfile
import threading
import time
from pathlib import Path
from typing import Any

import uvicorn
from fastapi.testclient import TestClient
from maxitor.repo_graph_port.cli import main as port_cli
from maxitor.repo_graph_port.server import create_app as create_port
from maxitor.web.api.app import create_app as create_maxitor
from maxitor.web.model.core.history_address import HistoryAddressError, parse_history

FOLDER = Path(tempfile.mkdtemp())
GRAPH_LINK = "https://github.com/owner/service/blob/main/graph.json"
FILE_LINK = "https://github.com/owner/service/blob/main/pyproject.toml"


def graph(entities: int) -> str:
    """A valid graph of ``entities`` nodes."""
    nodes = [{"id": f"shop.E{n}", "type": "Entity", "label": f"E{n}", "properties": {}} for n in range(entities)]
    return json.dumps({"schema_version": "1.2", "nodes": nodes, "edges": []}, indent=2) + "\n"


def component(number: str) -> str:
    """A component's ``version.toml``."""
    return f'kind = "component"\nversion = "{number}"\n'


def git(*arguments: str) -> None:
    """Run git in the example's repository."""
    subprocess.run(["git", "-C", str(FOLDER / "repo"), "-c", "user.name=T", "-c", "user.email=t@e", *arguments], check=True, capture_output=True)


def commit(message: str, files: dict[str, str]) -> None:
    """Write ``files`` and commit them."""
    for name, text in files.items():
        (FOLDER / "repo" / name).write_text(text, encoding="utf-8")
    git("add", "-A")
    git("commit", "-q", "-m", message)


def build_repository() -> None:
    """Five commits: model and number, code and number with a release, model only, code and number."""
    (FOLDER / "repo").mkdir()
    git("init", "-q", "-b", "main")
    commit("first", {"graph.json": graph(2), "version.toml": component("1.0.0")})
    commit("model and number", {"graph.json": graph(3), "version.toml": component("1.1.0")})
    commit("code and number", {"version.toml": component("1.1.1")})
    git("tag", "release-1.1.1")
    commit("model only", {"graph.json": graph(4)})
    commit("code, number", {"version.toml": component("1.1.2")})


def start_port() -> tuple[str, dict[str, str]]:
    """A real port over the repository, with three histories; its address and each history's key."""
    config = FOLDER / "histories.toml"
    entries = {"joined": 'version_file = "version.toml"\n', "plain": "", "broken": 'version_file = "pom.xml"\n'}
    commit("a file that holds no version", {"pom.xml": "<project/>"})
    config.write_text(
        "".join(f'[[history]]\nid = "{name}"\nhost = "local"\nrepository = "{FOLDER / "repo"}"\nbranch = "main"\ngraph_file = "graph.json"\n{extra}' for name, extra in entries.items()),
        encoding="utf-8",
    )
    keys = {}
    for name in entries:
        out = io.StringIO()
        with contextlib.redirect_stdout(out):
            port_cli(["--config", str(config), "key", "new", "--history", name, "--name", "anna"])
        keys[name] = out.getvalue().strip()
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        free = probe.getsockname()[1]
    server = uvicorn.Server(uvicorn.Config(create_port(config, FOLDER / "cache"), host="127.0.0.1", port=free, log_level="error"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.05)
    return f"http://127.0.0.1:{free}/api/v1/history", keys


def open_history(maxitor: TestClient, port: str, keys: dict[str, str], name: str) -> dict[str, Any]:
    """Open a history as the browser does: its name in the query, its key in the header."""
    answer: dict[str, Any] = maxitor.get("/api/history/open", params={"history": f"{port}/{name}"}, headers={"X-Maxitor-History-Key": keys[name]}).json()
    return answer


def drawn(point: dict[str, Any], file: bool) -> str:
    """How the track draws a point: its size, and the number under it."""
    size = "large" if point.get("graph_changed", True) else "small"
    number = point["version"] if file and point.get("version_changed") else "—"
    return f"{size}  {number}"


def show(history: dict[str, Any]) -> None:
    """One line per point, as the track draws it."""
    for point in history["versions"]:
        print(f'   {point["message"].strip().ljust(30)} {drawn(point, history["version_file"] is not None)}')


def flags_of(point: dict[str, Any]) -> list[str]:
    """The tags that are flags on the track."""
    return [tag["name"] for tag in point["tags"] if tag["release"] or tag["name"].startswith("release")]


def hidden_view(versions: list[dict[str, Any]], chosen: str | None) -> None:
    """What stays when the small points are hidden, as the track's own filter decides it."""
    kept = [point for point in versions if point.get("graph_changed", True) or point["sha"].startswith(chosen or "\0")]
    carried: dict[str, list[str]] = {}
    for point in versions:
        if point not in kept and flags_of(point):
            before = [each for each in kept if versions.index(each) < versions.index(point)]
            carried.setdefault((before[-1] if before else kept[0])["message"].strip(), []).extend(flags_of(point))
    print(f'   {", ".join(point["message"].strip() for point in kept)}')
    print(f"   flags on the points that stay: {carried}")


def same_graph(versions: list[dict[str, Any]], older: int, newer: int) -> bool:
    """Whether no point after the older one, up to the newer, changed the graph."""
    return all(point.get("graph_changed", True) is False for point in versions[older + 1 : newer + 1])

In [ ]:
build_repository()
port, keys = start_port()
maxitor = TestClient(create_maxitor())
maxitor.__enter__()
None

## A history with a version file

A large point changed the graph; a small one changed only the version. A number stands where the version rose, so `model only` is large with none.

In [ ]:
print("1. A history with a version file")
joined = open_history(maxitor, port, keys, "joined")
show(joined)
print(f'   version_file: {joined["version_file"]}, so the icon that hides small points is offered')

## No version file

The points it always had, every one large. There is nothing to hide, so no icon is offered.

In [ ]:
print("2. The same repository with no version file")
plain = open_history(maxitor, port, keys, "plain")
show(plain)
print(f'   version_file: {plain["version_file"]}, so no point is small and no icon is offered')

## What hiding leaves

Hiding drops the small points, except a chosen one, and their flags stand on the nearest point before them. What is chosen never changes.

In [ ]:
print("3. What hiding leaves")
versions = joined["versions"]
print("   nothing chosen:")
hidden_view(versions, None)
print('   "code and number" chosen, it stays while it is chosen:')
hidden_view(versions, versions[2]["sha"][:12])

## Two versions with one graph

No point between them changed the graph, so the comparison says so instead of showing an empty picture.

In [ ]:
print("4. Two versions with one graph")
print(f'   {versions[1]["message"].strip()} and {versions[2]["message"].strip()}: same graph {same_graph(versions, 1, 2)}')
print(f'   {versions[2]["message"].strip()} and {versions[3]["message"].strip()}: same graph {same_graph(versions, 2, 3)}')

## A version file that cannot be read

The graph opens, the track shows only the commits that changed the graph, and the reason reaches the foot of the picker.

In [ ]:
print("5. A version file that cannot be read")
broken = open_history(maxitor, port, keys, "broken")
print(f'   version_problem: {broken["version_problem"]}')
print(f'   {len(broken["versions"])} points, every one large: {all(point["graph_changed"] for point in broken["versions"])}')

## A public link with its version file

The link carries the file in its query, always in the same order, so one graph joined with two files is two histories.

In [ ]:
print("6. A public link with its version file")
history = parse_history(f"{GRAPH_LINK}?version_key=project.version&version_link={FILE_LINK}")
print(f"   name: {history.name}" if history else "   not a history")
for address in (f"{GRAPH_LINK}?branch=main", f"{GRAPH_LINK}?version_key=project.version"):
    try:
        parse_history(address)
    except HistoryAddressError as refused:
        print(f"   refused: {refused}")